## 🤖 AutoGen Exploration Notebook

This notebook explores **AutoGen**, a Microsoft open-source framework for building multi-agent AI systems that collaborate via conversations. It demonstrates core functionalities like agent conversations, multi-agent collaboration, code execution, tool integration, group chats, API integration, and custom agents.

## What is AutoGen?
AutoGen enables customizable agents powered by large language models (LLMs), tools, or human input, facilitating complex workflows through agent interactions for tasks like coding, data analysis, and automation.

## Objectives
- Introduce AutoGen's key features.
- Provide practical, beginner-friendly examples with theoretical context.
- Ensure valid JSON for the notebook to avoid parsing errors.

## Prerequisites
- Python 3.10+
- Install dependencies: `pip install autogen matplotlib yfinance requests`
- Optional: OpenAI API key (set as `OPENAI_API_KEY` environment variable) and OpenWeatherMap API key.
- Internet access for API calls.

## Structure
1. Setup and Basic Agent Conversation
2. Multi-Agent Collaboration
3. Code Execution with Agents
4. Tool Integration
5. Group Chat for Complex Tasks
6. Integration with External APIs
7. Custom Agent Creation

Let's get started!

## 1. Setup and Basic Agent Conversation

**Theory**: AutoGen agents, such as `AssistantAgent` and `UserProxyAgent`, communicate to perform tasks. The `UserProxyAgent` initiates conversations, while the `AssistantAgent` responds using an LLM.

In [ ]:
# Install required packages
!pip install autogen matplotlib yfinance requests

In [ ]:
# Basic Agent Conversation
import os
from autogen import AssistantAgent, UserProxyAgent

# Configure LLM
llm_config = {"model": "gpt-4", "api_key": os.environ.get("OPENAI_API_KEY", "your_openai_api_key")}

# Initialize agents
assistant = AssistantAgent("assistant", llm_config=llm_config)
user_proxy = UserProxyAgent("user_proxy", code_execution_config=False)

# Start conversation
user_proxy.initiate_chat(assistant, message="What is AutoGen, and how can it help with automation?")
print("Conversation completed. Check output above.")

## 2. Multi-Agent Collaboration

**Theory**: Multiple agents with specialized roles can collaborate to solve complex tasks, passing messages to refine outputs.

In [ ]:
# Multi-Agent Collaboration
from autogen import ConversableAgent

# Define agents with roles
analyst = ConversableAgent(
    "analyst",
    system_message="You are a data analyst. Provide insights based on user queries.",
    llm_config=llm_config,
    human_input_mode="NEVER"
)
reviewer = ConversableAgent(
    "reviewer",
    system_message="You review and refine the analyst's insights.",
    llm_config=llm_config,
    human_input_mode="NEVER"
)

# Start conversation
user_proxy.initiate_chat(
    analyst,
    message="Analyze the trend of AI adoption in businesses.",
    additional_recipients=[reviewer]
)
print("Multi-agent conversation completed.")

## 3. Code Execution with Agents

**Theory**: AutoGen agents can execute code to perform tasks like data analysis or visualization, using a code executor configured in the `UserProxyAgent`.

In [ ]:
# Code Execution
from autogen import UserProxyAgent
import autogen

# Configure code execution
user_proxy = UserProxyAgent(
    "user_proxy",
    code_execution_config={"executor": autogen.coding.LocalCommandLineCodeExecutor(work_dir="./coding")}
)

# Start conversation
user_proxy.initiate_chat(
    assistant,
    message="Plot a chart of Apple (AAPL) stock price for 2024 using yfinance and matplotlib. Save as aapl_plot.png."
)
print("Plot generated. Check aapl_plot.png in the 'coding' directory.")

## 4. Tool Integration

**Theory**: Agents can use external tools via function calling, allowing them to perform specific tasks like calculations or data processing.

In [ ]:
# Tool Integration
from autogen import register_function

# Define a simple tool
def calculator(a: float, b: float, operation: str) -> float:
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    else:
        raise ValueError("Unsupported operation. Use 'add' or 'subtract'.")

# Register tool
register_function(
    calculator,
    caller=assistant,
    executor=user_proxy,
    name="calculator",
    description="Performs basic arithmetic operations (add or subtract)."
)

# Test tool
user_proxy.initiate_chat(assistant, message="Calculate 10 + 5 using the calculator tool.")
print("Tool execution completed.")

## 5. Group Chat for Complex Tasks

**Theory**: Group chats enable multiple agents to collaborate dynamically, simulating a team working on a shared goal.

In [ ]:
# Group Chat
from autogen import GroupChat, GroupChatManager

# Define agents
planner = ConversableAgent("planner", system_message="Plan tasks and outline steps.", llm_config=llm_config)
coder = ConversableAgent("coder", system_message="Write and debug code.", llm_config=llm_config)
tester = ConversableAgent("tester", system_message="Test code for correctness.", llm_config=llm_config)

# Create group chat
group_chat = GroupChat(agents=[planner, coder, tester], messages=[])
manager = GroupChatManager(groupchat=group_chat, llm_config=llm_config)

# Start group chat
user_proxy.initiate_chat(manager, message="Develop a Python script to fetch and analyze stock data.")
print("Group chat completed.")

## 6. Integration with External APIs

**Theory**: Agents can call external APIs to fetch real-time data, enhancing their ability to provide relevant responses.

In [ ]:
# External API Integration
import requests
from autogen import register_function

# Define API tool
def fetch_weather(city: str) -> str:
    api_key = "your_weather_api_key"  # Replace with OpenWeatherMap API key
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    return response.json()["weather"][0]["description"]

# Register tool
register_function(
    fetch_weather,
    caller=assistant,
    executor=user_proxy,
    name="fetch_weather",
    description="Fetches weather data for a city."
)

# Test API call
user_proxy.initiate_chat(assistant, message="What's the weather in San Francisco?")
print("API integration completed.")

## 7. Custom Agent Creation

**Theory**: Users can create custom agents by extending AutoGen's base classes, tailoring behavior for specific tasks.

In [ ]:
# Custom Agent Creation
from autogen import ConversableAgent

# Define custom agent
class CustomSummaryAgent(ConversableAgent):
    def __init__(self, name, llm_config):
        super().__init__(
            name=name,
            system_message="Summarize text into concise bullet points.",
            llm_config=llm_config
        )

# Initialize custom agent
summarizer = CustomSummaryAgent("summarizer", llm_config=llm_config)

# Test custom agent
user_proxy.initiate_chat(
    summarizer,
    message="Summarize: AutoGen is a framework for building multi-agent systems, enabling collaboration among LLMs, tools, and humans for tasks like coding and automation."
)
print("Custom agent conversation completed.")

## Conclusion

This notebook demonstrated AutoGen's core functionalities, from basic agent conversations to custom agent creation. For more advanced features, explore the [AutoGen Documentation](https://microsoft.github.io/autogen/).